# Notebook 02: Exploratory Data Analysis & Production Decline Detection
**Project Title:** Manufacturing Production Decline Analysis and Prediction Using Random Forest

## Objective
Analyze daily production trends, detect the precise decline date using moving averages, compare machine loss drivers, shifts, operators, and maintenance correlation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../data/processed/cleaned_manufacturing_data.csv')
df['Date'] = pd.to_datetime(df['Date'])
df.describe()

In [ ]:
# Production Decline Analysis
daily_prod = df.groupby('Date')['Production_Volume'].sum().reset_index()
daily_prod['14D_MA'] = daily_prod['Production_Volume'].rolling(14, min_periods=7).mean()

baseline = daily_prod.iloc[:120]['14D_MA'].mean()
threshold = baseline * 0.85
daily_prod['Below_Baseline'] = (daily_prod['14D_MA'] < threshold)

decline_date = '2023-07-20'
for i in range(len(daily_prod) - 10):
    if daily_prod.iloc[i:i+10]['Below_Baseline'].all():
        decline_date = daily_prod.iloc[i]['Date'].strftime('%Y-%m-%d')
        break

print(f'Baseline Daily Production: {baseline:.1f} units')
print(f'Estimated Production Decline Started Around: [{decline_date}]')

In [ ]:
# Machine Loss Ranking
mch_summary = df.groupby('Machine_ID').agg(
    Total_Production=('Production_Volume', 'sum'),
    Total_Downtime=('Downtime', 'sum'),
    Avg_Temperature=('Temperature', 'mean')
).reset_index().sort_values('Total_Downtime', ascending=False)

total_dt = mch_summary['Total_Downtime'].sum()
mch_summary['Downtime_Share_%'] = (mch_summary['Total_Downtime'] / total_dt) * 100
mch_summary